# Label database exports from Google Drive

Load the public code from GitHub and keep exports/results in your private Drive folder. Run the cells in order on a CPU runtime. Labels use the existing keyword rules; no model API or database credentials are required.

Setup and supported formats: [Colab workflow documentation](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/default/docs_agent/colab.md).


## Load the code


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile

CODE_URL = "https://github.com/gettodaze/OMSCS_ITR_QMatrix.git"
CODE_REF = "default"  # Branch or tag; use a release tag to pin a published version.
CODE_ROOT = Path(tempfile.mkdtemp(prefix="qmatrix-code-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, CODE_URL, str(CODE_ROOT)],
               check=True, capture_output=True, text=True)
CODE_REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE_ROOT, text=True).strip()
sys.path.insert(0, str(CODE_ROOT))
for name in ("parse_records", "prepare_rayyan", "label_exports"):
    sys.modules.pop(name, None)
print("Code revision:", CODE_REVISION)


## Connect Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Choose the private data folder

Create `QMatrix/exports/<database>/` in My Drive and put the native downloads in the database folders. Supported files: `.ris`, `.nbib`, `.enw`, and tagged PubMed/RIS/EndNote `.txt`. The old candidate CSV is optional, used only for comparison. Raw provider CSV/BibTeX files are not parsed by this workflow.


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/QMatrix")  # Edit this if your folder is elsewhere.
EXPORT_ROOT = DATA_ROOT / "exports"
DRIVE_OUTPUT_ROOT = DATA_ROOT / "output"
BASELINE_FILE = ""  # Optional, e.g. "input.csv", relative to DATA_ROOT.
COUNTS_FILE = ""    # Optional, e.g. "export_counts.csv", relative to DATA_ROOT.

if not EXPORT_ROOT.is_dir():
    raise FileNotFoundError("Create the exports/<database>/ folders in your chosen Drive data folder first.")
for name in (BASELINE_FILE, COUNTS_FILE):
    if name and not (DATA_ROOT / name).is_file():
        raise FileNotFoundError(DATA_ROOT / name)
print("Data folder selected. Original downloads will remain unchanged.")


## Prepare native files and check counts


In [ ]:
from datetime import datetime, timezone
from prepare_rayyan import prepare

RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%f_UTC")
WORK_ROOT = Path(tempfile.mkdtemp(prefix="qmatrix-data-"))
SNAPSHOT = WORK_ROOT / "exports"
shutil.copytree(EXPORT_ROOT, SNAPSHOT)

def local_optional(name):
    if not name:
        return None
    destination = Path(tempfile.mkdtemp(prefix="optional-", dir=WORK_ROOT)) / Path(name).name
    shutil.copy2(DATA_ROOT / name, destination)
    return destination

PREPARED_ROOT = WORK_ROOT / "prepared"
audit = prepare(SNAPSHOT, PREPARED_ROOT,
                baseline=local_optional(BASELINE_FILE), counts=local_optional(COUNTS_FILE))
print("Exported records:", sum(item["records"] for item in audit["files"]))
if any(item["count_matches"] is False for item in audit["files"]):
    failed_output = DRIVE_OUTPUT_ROOT / (RUN_ID + "_count_mismatch")
    shutil.copytree(PREPARED_ROOT, failed_output)
    raise ValueError(f"Export counts do not match. Audit and native files saved to {failed_output}; check before labeling/importing.")


## Suggest labels


In [ ]:
from label_exports import label_exports

LABELS_ROOT = WORK_ROOT / "labels"
summary = label_exports(PREPARED_ROOT / "originals", LABELS_ROOT)
assert summary["records"] == sum(item["records"] for item in audit["files"])
print("Labeled records:", summary["records"])
print("Records missing abstracts:", summary["without_abstract"])
print("Records without a suggested label:", summary["without_suggested_labels"])


## Save results to Drive


In [ ]:
RUN_OUTPUT = DRIVE_OUTPUT_ROOT / RUN_ID
RUN_OUTPUT.mkdir(parents=True, exist_ok=False)
shutil.copytree(PREPARED_ROOT, RUN_OUTPUT / "prepared")
shutil.copytree(LABELS_ROOT, RUN_OUTPUT / "labels")
(RUN_OUTPUT / "run_metadata.json").write_text(json.dumps({
    "code_url": CODE_URL, "code_ref": CODE_REF, "code_revision": CODE_REVISION,
    "run_id": RUN_ID, "baseline_used": bool(BASELINE_FILE), "counts_used": bool(COUNTS_FILE),
    "labeling_method": summary["labeling_method"], "records": summary["records"]
}, indent=2) + "\n", encoding="utf-8")
print("Saved results to:", RUN_OUTPUT)


## Review aggregate results


In [ ]:
print("Records by database:")
for name, count in summary["by_database"].items():
    print(f"  {name}: {count}")
print("Records by suggested label:")
for name, count in summary["by_label"].items():
    print(f"  {name}: {count}")


## Files to use

- `labels/labelled_records.csv`: suggested labels, common citation fields, and every original tag in format-specific columns (repeated values are JSON lists).
- `labels/labelled_records.jsonl`: source fields, labels and provenance retained as structured records.
- `prepared/audit.json`: counts, document types, metadata coverage and optional baseline comparison.
- `prepared/by_database/`: native concatenated exports for named Rayyan imports. Do not also import `prepared/combined.*`, which contains the same records.

Labels are suggestions, not screening decisions. Rayyan may ignore extra CSV columns; automatic creation of Rayyan labels from these notes has not been verified. Test a small import first. The preserved native files and JSON retain metadata independently of Rayyan.

The shared notebook contains no data outputs. Keep executed copies with any record previews private. See [data sharing guidance](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/default/docs_agent/data_sharing.md).
